# Рекомендательная система фильмов на основе Node2Vec

В этом ноутбуке строится граф фильмов по данным MovieLens 100k: фильмы являются вершинами, а вес ребра показывает, сколько пользователей высоко оценили оба фильма. Затем на графе обучаются Node2Vec-эмбеддинги, которые используются для поиска похожих фильмов.

## 1. Подготовка окружения

Ноутбук рассчитан на запуск в отдельном виртуальном окружении. Если зависимости отсутствуют, создайте окружение в папке проекта и выберите его как Jupyter kernel:

```bash
python3 -m venv .venv
.venv/bin/python -m pip install -U pip
.venv/bin/python -m pip install pandas numpy networkx node2vec gensim scikit-learn matplotlib tqdm ipykernel
.venv/bin/python -m ipykernel install --user --name film-node2vec --display-name "Python (film-node2vec)"
```

In [ ]:
import importlib.util
import sys

required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "networkx": "networkx",
    "node2vec": "node2vec",
    "gensim": "gensim",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
}

missing = [pkg for module, pkg in required_packages.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Missing packages:", ", ".join(missing))
    print("Install them in this kernel environment, for example:")
    print(f"{sys.executable} -m pip install " + " ".join(missing))
    raise ModuleNotFoundError("Install missing packages and restart this notebook kernel.")
else:
    print("All required packages are available.")

In [ ]:
from collections import Counter
from itertools import combinations
from pathlib import Path
import random
import sys
import warnings

import numpy as np
import pandas as pd

# Compatibility guard for some gensim/scipy combinations.
import scipy.linalg as scipy_linalg
if not hasattr(scipy_linalg, "triu"):
    scipy_linalg.triu = np.triu

import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
    warnings.filterwarnings("ignore", message="FigureCanvasAgg is non-interactive.*")
import matplotlib.pyplot as plt
import networkx as nx
from node2vec import Node2Vec

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("ml-100k")
RATINGS_PATH = DATA_DIR / "u.data"
ITEMS_PATH = DATA_DIR / "u.item"
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

MIN_RATING = 4
MIN_COMMON_USERS = 20

## 2. Загрузка MovieLens 100k

Файл `u.data` содержит оценки пользователей, а `u.item` - названия фильмов и one-hot признаки жанров.

In [ ]:
rating_columns = ["user_id", "movie_id", "rating", "timestamp"]
ratings = pd.read_csv(RATINGS_PATH, sep="\t", names=rating_columns)

movie_genres = [
    "unknown", "Action", "Adventure", "Animation", "Children's", "Comedy", "Crime",
    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery",
    "Romance", "Sci-Fi", "Thriller", "War", "Western",
]
item_columns = ["movie_id", "title", "release_date", "video_release_date", "imdb_url", *movie_genres]
movies = pd.read_csv(ITEMS_PATH, sep="|", names=item_columns, encoding="latin-1")

ratings = ratings.drop_duplicates(subset=["user_id", "movie_id"])
movies = movies.drop_duplicates(subset=["movie_id"]).set_index("movie_id", drop=False)

print(f"Ratings: {len(ratings):,}")
print(f"Users: {ratings['user_id'].nunique():,}")
print(f"Movies with ratings: {ratings['movie_id'].nunique():,}")
print(f"Movies metadata rows: {len(movies):,}")
ratings.head()

## 3. Построение графа фильмов

Оставляем только положительные взаимодействия (`rating >= 4`). Для каждого пользователя перебираем все пары понравившихся фильмов и считаем, сколько пользователей положительно оценили оба фильма.

In [ ]:
def build_movie_graph(ratings_df: pd.DataFrame, min_rating: int = 4, min_common_users: int = 20) -> tuple[nx.Graph, pd.DataFrame]:
    """Build a weighted movie co-like graph from user ratings."""
    positive = ratings_df.loc[ratings_df["rating"] >= min_rating, ["user_id", "movie_id"]]

    pair_counts: Counter[tuple[int, int]] = Counter()
    for _, user_items in positive.groupby("user_id"):
        liked_movies = sorted(user_items["movie_id"].unique())
        pair_counts.update(combinations(liked_movies, 2))

    edges = [
        {"source": source, "target": target, "weight": weight}
        for (source, target), weight in pair_counts.items()
        if weight >= min_common_users
    ]
    edges_df = pd.DataFrame(edges).sort_values("weight", ascending=False).reset_index(drop=True)

    graph = nx.Graph()
    if not edges_df.empty:
        graph.add_weighted_edges_from(edges_df[["source", "target", "weight"]].itertuples(index=False, name=None))
    nx.set_node_attributes(graph, movies["title"].to_dict(), "title")

    return graph, edges_df

G, edges_df = build_movie_graph(ratings, MIN_RATING, MIN_COMMON_USERS)
print(f"Positive ratings (rating >= {MIN_RATING}): {len(ratings[ratings['rating'] >= MIN_RATING]):,}")
print(f"Graph nodes: {G.number_of_nodes():,}")
print(f"Graph edges: {G.number_of_edges():,}")
edges_df.head()

In [ ]:
def graph_size_for_threshold(min_common_users: int) -> dict[str, int]:
    graph, _ = build_movie_graph(ratings, MIN_RATING, min_common_users)
    return {
        "min_common_users": min_common_users,
        "nodes": graph.number_of_nodes(),
        "edges": graph.number_of_edges(),
    }

threshold_report = pd.DataFrame(graph_size_for_threshold(t) for t in [20, 25, 30, 31, 32, 33, 34, 35, 40, 45, 50])
threshold_report

> Примечание: в тексте задания указаны ожидаемые `410` узлов и `14 936` ребер. При прямом применении условий `rating >= 4` и `min_common_users >= 20` к полному `u.data` получается более плотный граф. Поэтому основной эксперимент оставлен строго по формальным условиям задания, а таблица выше показывает чувствительность размера графа к порогу совместных положительных оценок.

## 4. Анализ графа

Посмотрим на связность графа, веса ребер и фильмы с наибольшей взвешенной степенью.

In [ ]:
components = list(nx.connected_components(G))
edge_weights = np.array([data["weight"] for _, _, data in G.edges(data=True)])
weighted_degree = dict(G.degree(weight="weight"))

print(f"Connected components: {len(components)}")
print(f"Largest component size: {len(max(components, key=len)):,}")
print(f"Average edge weight: {edge_weights.mean():.2f}")
print(f"Median edge weight: {np.median(edge_weights):.2f}")
print(f"Max edge weight: {edge_weights.max():.0f}")

top_connected = (
    pd.DataFrame({"movie_id": list(weighted_degree.keys()), "weighted_degree": list(weighted_degree.values())})
    .join(movies[["title"]], on="movie_id")
    .sort_values("weighted_degree", ascending=False)
    .head(15)
)
top_connected

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
degrees = [degree for _, degree in G.degree()]
plt.hist(degrees, bins=30, color="#4C72B0", edgecolor="white")
plt.title("Degree distribution")
plt.xlabel("Degree")
plt.ylabel("Number of movies")

plt.subplot(1, 2, 2)
plt.hist(edge_weights, bins=30, color="#55A868", edgecolor="white")
plt.title("Edge weight distribution")
plt.xlabel("Common positive ratings")
plt.ylabel("Number of edges")

plt.tight_layout()
plt.show()

In [ ]:
# Visualize a compact ego-subgraph around a well-connected movie.
center_movie_id = int(top_connected.iloc[0]["movie_id"])
neighbors = sorted(G.neighbors(center_movie_id), key=lambda node: G[center_movie_id][node]["weight"], reverse=True)[:25]
subgraph_nodes = [center_movie_id, *neighbors]
H = G.subgraph(subgraph_nodes).copy()

plt.figure(figsize=(12, 8))
pos = nx.spring_layout(H, seed=RANDOM_STATE, weight="weight")
node_sizes = [350 + 12 * H.degree(node, weight="weight") for node in H.nodes()]
edge_widths = [0.5 + H[u][v]["weight"] / 30 for u, v in H.edges()]
labels = {node: movies.loc[node, "title"][:28] for node in H.nodes()}

nx.draw_networkx_nodes(H, pos, node_size=node_sizes, node_color="#4C72B0", alpha=0.85)
nx.draw_networkx_edges(H, pos, width=edge_widths, alpha=0.35)
nx.draw_networkx_labels(H, pos, labels=labels, font_size=8)
plt.title(f"Ego-subgraph: {movies.loc[center_movie_id, 'title']}")
plt.axis("off")
plt.show()

## 5. Обучение Node2Vec

Node2Vec генерирует случайные блуждания по графу и обучает модель Word2Vec, где роль слов играют идентификаторы фильмов. Вес ребра используется как вероятность перехода: чем больше пользователей совместно оценили пару фильмов, тем чаще случайное блуждание будет переходить между ними.

In [ ]:
node2vec_params = {
    "dimensions": 64,
    "walk_length": 30,
    "num_walks": 100,
    "p": 1.0,
    "q": 1.0,
    "workers": 2,
    "weight_key": "weight",
    "seed": RANDOM_STATE,
}
word2vec_params = {
    "window": 10,
    "min_count": 1,
    "batch_words": 256,
    "seed": RANDOM_STATE,
}

node2vec = Node2Vec(G, **node2vec_params)
model = node2vec.fit(**word2vec_params)
print("Vocabulary size:", len(model.wv.index_to_key))
print("Embedding dimension:", model.wv.vector_size)

In [ ]:
embedding_rows = []
for node_id in model.wv.index_to_key:
    movie_id = int(node_id)
    embedding_rows.append({
        "movie_id": movie_id,
        "title": movies.loc[movie_id, "title"],
        **{f"emb_{i}": value for i, value in enumerate(model.wv[node_id])},
    })

embeddings = pd.DataFrame(embedding_rows).sort_values("movie_id").reset_index(drop=True)
embeddings_path = ARTIFACTS_DIR / "movie_node2vec_embeddings.csv"
embeddings.to_csv(embeddings_path, index=False)
print(f"Saved embeddings to {embeddings_path}")
embeddings.head()

## 6. Рекомендации похожих фильмов

Функция ниже принимает часть названия фильма, находит соответствующий `movie_id`, затем возвращает ближайшие фильмы в пространстве Node2Vec-эмбеддингов.

In [ ]:
def movie_genre_labels(movie_id: int) -> str:
    row = movies.loc[movie_id, movie_genres]
    labels = [genre for genre in movie_genres if row[genre] == 1]
    return ", ".join(labels) if labels else "unknown"


def find_movies(title_query: str, limit: int = 10) -> pd.DataFrame:
    mask = movies["title"].str.contains(title_query, case=False, regex=False, na=False)
    return movies.loc[mask, ["movie_id", "title"]].head(limit)


def resolve_movie_id(title_query: str) -> int:
    matches = find_movies(title_query, limit=20)
    if matches.empty:
        raise ValueError(f"Movie not found: {title_query}")

    query = title_query.lower().strip()
    exact = matches[matches["title"].str.lower() == query]
    selected = exact.iloc[0] if not exact.empty else matches.iloc[0]
    movie_id = int(selected["movie_id"])

    if str(movie_id) not in model.wv:
        raise ValueError(
            f"Movie '{selected['title']}' is not present in the Node2Vec graph. "
            "Try a more popular movie or lower MIN_COMMON_USERS."
        )
    return movie_id


def recommend_movies(title_query: str, top_k: int = 10) -> pd.DataFrame:
    source_id = resolve_movie_id(title_query)
    source_key = str(source_id)
    similar = model.wv.most_similar(source_key, topn=top_k * 3)

    recommendations = []
    for neighbor_key, similarity in similar:
        movie_id = int(neighbor_key)
        if movie_id == source_id:
            continue
        recommendations.append({
            "movie_id": movie_id,
            "title": movies.loc[movie_id, "title"],
            "genres": movie_genre_labels(movie_id),
            "similarity": similarity,
            "edge_weight_to_source": G[source_id][movie_id]["weight"] if G.has_edge(source_id, movie_id) else 0,
        })
        if len(recommendations) == top_k:
            break

    source_title = movies.loc[source_id, "title"]
    print(f"Source movie: {source_title} ({movie_genre_labels(source_id)})")
    return pd.DataFrame(recommendations)

In [ ]:
find_movies("Star Wars")

In [ ]:
recommend_movies("Star Wars", top_k=10)

In [ ]:
examples = ["Toy Story", "Fargo", "Return of the Jedi", "Scream"]
for title in examples:
    print("=" * 90)
    display(recommend_movies(title, top_k=5))

## 7. Влияние параметров `p` и `q`

В Node2Vec параметр `p` управляет вероятностью немедленного возврата к предыдущей вершине, а `q` задаёт баланс между BFS-подобным и DFS-подобным обходом.

- Низкий `q` сильнее поощряет уход от текущей окрестности и может находить более дальние структурные аналогии.
- Высокий `q` удерживает блуждание рядом с исходной областью графа и обычно даёт более локальные рекомендации.
- Низкий `p` повышает вероятность возврата назад, высокий `p` делает такие возвраты менее вероятными.

In [ ]:
def train_node2vec_model(p: float, q: float, dimensions: int = 32, num_walks: int = 60):
    n2v = Node2Vec(
        G,
        dimensions=dimensions,
        walk_length=25,
        num_walks=num_walks,
        p=p,
        q=q,
        workers=2,
        weight_key="weight",
        seed=RANDOM_STATE,
    )
    return n2v.fit(window=8, min_count=1, batch_words=256, seed=RANDOM_STATE)


def recommendations_from_model(trained_model, title_query: str, top_k: int = 5) -> list[str]:
    movie_id = resolve_movie_id(title_query)
    neighbors = trained_model.wv.most_similar(str(movie_id), topn=top_k)
    return [movies.loc[int(neighbor), "title"] for neighbor, _ in neighbors]

parameter_sets = [
    {"name": "balanced", "p": 1.0, "q": 1.0},
    {"name": "BFS-like / local", "p": 1.0, "q": 2.0},
    {"name": "DFS-like / exploratory", "p": 1.0, "q": 0.5},
    {"name": "less backtracking", "p": 2.0, "q": 1.0},
]

comparison_rows = []
probe_title = "Star Wars"
for params in parameter_sets:
    print(f"Training {params['name']} model: p={params['p']}, q={params['q']}")
    variant_model = train_node2vec_model(p=params["p"], q=params["q"])
    comparison_rows.append({
        "setting": params["name"],
        "p": params["p"],
        "q": params["q"],
        "top_recommendations": " | ".join(recommendations_from_model(variant_model, probe_title, top_k=5)),
    })

pd.DataFrame(comparison_rows)

## 8. Простая интерпретация качества

Для быстрой проверки качества посмотрим, насколько рекомендации совпадают с исходным фильмом по жанрам. Это не полноценная offline-оценка, но полезная sanity check-метрика для интерпретации результата.

In [ ]:
def genre_set(movie_id: int) -> set[str]:
    row = movies.loc[movie_id, movie_genres]
    return {genre for genre in movie_genres if row[genre] == 1}


def genre_jaccard(source_id: int, target_id: int) -> float:
    source_genres = genre_set(source_id)
    target_genres = genre_set(target_id)
    if not source_genres and not target_genres:
        return 1.0
    return len(source_genres & target_genres) / len(source_genres | target_genres)


def recommendation_quality_snapshot(title_query: str, top_k: int = 10) -> pd.DataFrame:
    source_id = resolve_movie_id(title_query)
    recs = recommend_movies(title_query, top_k=top_k)
    recs["genre_jaccard"] = recs["movie_id"].apply(lambda target_id: genre_jaccard(source_id, target_id))
    return recs

quality_snapshot = recommendation_quality_snapshot("Star Wars", top_k=10)
print(f"Average genre Jaccard: {quality_snapshot['genre_jaccard'].mean():.3f}")
quality_snapshot

## Выводы

1. Графовый подход переводит пользовательские предпочтения в структуру связей между фильмами: чем чаще два фильма получают высокие оценки от одних и тех же пользователей, тем сильнее ребро между ними.
2. Node2Vec позволяет получить плотные векторные представления фильмов, в которых близость отражает не только прямое совместное потребление, но и положение фильма в графе похожих предпочтений.
3. Параметры `p` и `q` заметно влияют на рекомендации: локальные обходы дают более близкие по соседству фильмы, а исследовательские обходы могут находить менее очевидные структурные аналоги.
4. Такой подход хорошо подходит для рекомендаций похожих фильмов, но для production-системы его стоит дополнить временным train/test-разделением, ranking-метриками (`Precision@K`, `Recall@K`, `NDCG@K`) и обработкой холодного старта.